In [2]:
# ======= Variables =========

# Fill in the variables below and run cell first
# Note: Use forward slashes / instead of backslashes \ in file paths

DATASET_DIR = "../dataset"
EXCEL_FILE = f"{DATASET_DIR}/CAASE Exercise - Mar 2026.xlsx"
OUTPUT_DIR = f"{DATASET_DIR}/original_sheets"


In [ ]:
# Split initial Excel file into separate CSVs for each sheet

from pathlib import Path
import re
import pandas as pd
    
excel_path = Path(EXCEL_FILE)  

workbook = pd.ExcelFile(excel_path, engine="openpyxl")

for sheet_name in workbook.sheet_names:
    df = pd.read_excel(workbook, sheet_name=sheet_name)
    safe_sheet_name = re.sub(r"[^A-Za-z0-9._-]+", "_", sheet_name).strip("_") or "sheet"
    out_file = Path(OUTPUT_DIR) / f"{safe_sheet_name}.csv"
    df.to_csv(out_file, index=False)

print(f"Exported {len(workbook.sheet_names)} sheets to: {Path(OUTPUT_DIR).resolve()}")


In [6]:
# Load each saved sheet into a pandas DataFrame and inspect its shape

from pathlib import Path
import pandas as pd

output_path = Path(OUTPUT_DIR)
dataframes = {}

for csv_file in sorted(output_path.glob("*.csv")):
    df = pd.read_csv(csv_file)
    dataframes[csv_file.stem] = df

    print(f"{csv_file.name}")
    print(f"  rows: {df.shape[0]}")
    print(f"  columns: {df.shape[1]}")
    print(f"  column names: {list(df.columns)}")
    print()


Data_Dictionary.csv
  rows: 15
  columns: 5
  column names: ['Table Name', 'Column Name', 'Data Type', 'Values/Format', 'Descriptions']

dim_patient.csv
  rows: 4020
  columns: 3
  column names: ['PATIENT_ID', 'BIRTH_YEAR', 'GENDER']

dim_physician.csv
  rows: 25343
  columns: 5
  column names: ['PHYSICIAN_ID', 'STATE', 'PHYSICIAN_TYPE', 'GENDER', 'BIRTH_YEAR']

fact_txn.csv
  rows: 115274
  columns: 7
  column names: ['TXN_DT', 'PATIENT_ID', 'PHYSICIAN_ID', 'TXN_LOCATION_TYPE', 'INSURANCE_TYPE', 'TXN_TYPE', 'TXN_DESC']

Instructions.csv
  rows: 53
  columns: 2
  column names: ['Unnamed: 0', 'Unnamed: 1']

model_table.csv
  rows: 9
  columns: 4
  column names: ['Column Name', 'Data Type', 'Values/Format', 'Descriptions']



In [3]:
# Read the dim_patient, dim_physician and fact_txn DataFrames

from pathlib import Path
import pandas as pd

output_path = Path(OUTPUT_DIR)

dim_patient = pd.read_csv(output_path / "dim_patient.csv")
dim_physician = pd.read_csv(output_path / "dim_physician.csv")
fact_txn = pd.read_csv(output_path / "fact_txn.csv")

print(f"dim_patient: rows={dim_patient.shape[0]}, columns={dim_patient.shape[1]}")
print(f"dim_physician: rows={dim_physician.shape[0]}, columns={dim_physician.shape[1]}")
print(f"fact_txn: rows={fact_txn.shape[0]}, columns={fact_txn.shape[1]}")

dim_patient: rows=4020, columns=3
dim_physician: rows=25343, columns=5
fact_txn: rows=115274, columns=7


In [25]:
# Explore unique values for dim_patient
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from ml_antiviral_diagnosis import summarize_unique_values

dim_patient_eda_df = summarize_unique_values(dim_patient)
dim_patient_eda_df

,column,dtype,unique_count,unique_values
0,PATIENT_ID,int64,4020,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,BIRTH_YEAR,int64,86,"[1988, 2020, 1973, 2022, 1951, 2021, 1977, 199..."
2,GENDER,str,2,"[M, F]"


In [14]:
# Explore unique values for dim_physician
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from ml_antiviral_diagnosis import summarize_unique_values

dim_physician_eda_df = summarize_unique_values(dim_physician)

In [17]:
dim_physician_eda_df

,column,dtype,unique_count,unique_values
0,PHYSICIAN_ID,int64,25343,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,STATE,str,58,"[TX, IN, CA, WA, OH, MS, SC, NJ, TN, MO, PA, A..."
2,PHYSICIAN_TYPE,str,199,"[PHYSICAL THERAPY, PHYSICIAN ASSISTANT, EMERGE..."
3,GENDER,str,3,"[nan, F, M]"
4,BIRTH_YEAR,float64,69,"[nan, 1990.0, 1989.0, 1991.0, 1988.0, 1984.0, ..."


In [ ]:
# Peruse the unique values of a specific column
column_name = "PHYSICIAN_TYPE"
values = dim_physician_eda_df.loc[dim_physician_eda_df["column"] == column_name, "unique_values"].iloc[0]
values

['PHYSICAL THERAPY',
 'PHYSICIAN ASSISTANT',
 'EMERGENCY MEDICINE',
 'NURSE PRACTITIONER',
 'FAMILY MEDICINE',
 'CASE MANAGER / CARE COORDINATOR',
 'CHIROPRACTIC',
 'CERTIFIED NURSE ANESTHETIST',
 'OCCUPATIONAL THERAPY',
 'PSYCHOLOGY',
 'DENTIST',
 'CLINICAL SOCIAL WORKER',
 'BEHAVIORAL HEALTH & SOCIAL SERVICES',
 'STUDENT, HEALTH CARE',
 'INTERNAL MEDICINE',
 'INTERVENTIONAL CARDIOLOGY',
 'GENERAL PRACTICE',
 'REGISTERED DIETICIAN',
 'OPTOMETRIST',
 'DIAGNOSTIC RADIOLOGY',
 'NEUROLOGY',
 'MOLECULAR GENETIC PATHOLOGY (PATHOLOGY)',
 'ANCILLARY SERVICES',
 'RADIATION ONCOLOGY',
 'THORACIC SURGERY',
 'PEDIATRICS',
 'PERFUSIONIST',
 'OPHTHALMOLOGY',
 'PSYCHIATRY',
 'PLASTIC SURGERY',
 'DERMATOLOGY',
 'ORTHOPEDIC SURGERY',
 'GENERAL SURGERY',
 'CLINICAL CARDIAC ELECTROPHYSIOLOGY',
 'CARDIOVASCULAR DISEASE',
 'HOSPITALIST',
 'PEDIATRIC CRITICAL CARE MEDICINE',
 'NUCLEAR MEDICINE',
 'TECHNOLOGIST - AUDIOLOGY/SPEECH/LANGUAGE',
 'PODIATRIST',
 'ANESTHESIOLOGY',
 'PEDIATRIC OTOLARYNGOLOGY',
 'PH

In [ ]:
# how many rows in dim_physician have PHYSICIAN_TYPE value "UNSPECIFIED"
unspecified_count = dim_physician.loc[dim_physician["PHYSICIAN_TYPE"] == "UNSPECIFIED"].shape[0]
unspecified_count

4

In [13]:
dim_physician

,PHYSICIAN_ID,STATE,PHYSICIAN_TYPE,GENDER,BIRTH_YEAR
0,1,TX,PHYSICAL THERAPY,NaN,NaN
1,2,IN,PHYSICIAN ASSISTANT,NaN,NaN
2,3,CA,EMERGENCY MEDICINE,NaN,NaN
3,4,TX,NURSE PRACTITIONER,NaN,NaN
4,5,WA,EMERGENCY MEDICINE,NaN,NaN
...,...,...,...,...,...
25338,12730,FL,FAMILY MEDICINE,F,1981.0
25339,12731,FL,INTERVENTIONAL CARDIOLOGY,M,1975.0
25340,12732,CA,OPHTHALMOLOGY,M,1973.0
25341,12733,MI,GENERAL SURGERY,M,1975.0


In [23]:
# Explore unique values for fact_txn
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from ml_antiviral_diagnosis import summarize_unique_values

fact_txn_eda_df = summarize_unique_values(fact_txn)

In [32]:
# Compare PATIENT_ID and PHYSICIAN_ID unique values across tables
from importlib import reload
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
import ml_antiviral_diagnosis.eda as eda

reload(eda)


def get_unique_values(eda_df, column_name):
    return eda_df.loc[eda_df["column"] == column_name, "unique_values"].iloc[0]


dim_patient_ids = get_unique_values(dim_patient_eda_df, "PATIENT_ID")
fact_txn_patient_ids = get_unique_values(fact_txn_eda_df, "PATIENT_ID")
dim_physician_ids = get_unique_values(dim_physician_eda_df, "PHYSICIAN_ID")
fact_txn_physician_ids = get_unique_values(fact_txn_eda_df, "PHYSICIAN_ID")

print("PATIENT_ID: dim_patient -> fact_txn")
patient_forward_result = eda.count_unique_values_only_in_first(
    dim_patient_ids,
    fact_txn_patient_ids,
)
print()

print("PATIENT_ID: fact_txn -> dim_patient")
patient_reverse_result = eda.count_unique_values_only_in_first(
    fact_txn_patient_ids,
    dim_patient_ids,
)
print()

print("PHYSICIAN_ID: dim_physician -> fact_txn")
physician_forward_result = eda.count_unique_values_only_in_first(
    dim_physician_ids,
    fact_txn_physician_ids,
)
print()

print("PHYSICIAN_ID: fact_txn -> dim_physician")
physician_reverse_result = eda.count_unique_values_only_in_first(
    fact_txn_physician_ids,
    dim_physician_ids,
)

{
    "patient_forward_result": patient_forward_result,
    "patient_reverse_result": patient_reverse_result,
    "physician_forward_result": physician_forward_result,
    "physician_reverse_result": physician_reverse_result,
}

PATIENT_ID: dim_patient -> fact_txn
Unique values only in first list: 0 found -> []

PATIENT_ID: fact_txn -> dim_patient
Unique values only in first list: 0 found -> []

PHYSICIAN_ID: dim_physician -> fact_txn
Unique values only in first list: 0 found -> []

PHYSICIAN_ID: fact_txn -> dim_physician
Unique values only in first list: 1 found -> [nan]


{'patient_forward_result': {'count': 0, 'values': []},
 'patient_reverse_result': {'count': 0, 'values': []},
 'physician_forward_result': {'count': 0, 'values': []},
 'physician_reverse_result': {'count': 1, 'values': [nan]}}

In [35]:
# Peruse the unique values of a specific column
column_name = "TXN_TYPE"
values = fact_txn_eda_df.loc[fact_txn_eda_df["column"] == column_name, "unique_values"].iloc[0]
values

['SYMPTOMS', 'CONDITIONS', 'CONTRAINDICATIONS', 'TREATMENTS']

In [24]:
fact_txn_eda_df

,column,dtype,unique_count,unique_values
0,TXN_DT,str,2030,"[2021-12-27, 2020-02-25, 2022-01-25, 2022-03-0..."
1,PATIENT_ID,int64,4020,"[3124, 3352, 2902, 2943, 3589, 1459, 2414, 102..."
2,PHYSICIAN_ID,float64,25344,"[nan, 21972.0, 21586.0, 2439.0, 6381.0, 22709...."
3,TXN_LOCATION_TYPE,str,49,"[INDEPENDENT LABORATORY, OFFICE, HOSPITAL OUTP..."
4,INSURANCE_TYPE,str,5,"[COMMERCIAL, MEDICARE, MEDICAID, UNSPECIFIED, ..."
5,TXN_TYPE,str,4,"[SYMPTOMS, CONDITIONS, CONTRAINDICATIONS, TREA..."
6,TXN_DESC,str,53,"[ACUTE_PHARYNGITIS, obesity, immunocompromised..."
